# Eksplorasi & Akuisisi Data Akomodasi di Kecamatan Kintamani
Proyek praktikum pengumpulan data akomodasi dari OpenStreetMap.

In [ ]:
# Akuisisi Data Akomodasi Kintamani via Overpass API
import json
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import requests

# 1. Konfigurasi Endpoint & Header
URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (2501010134@instiki.ac.id)"}

QUERY = """
[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;
"""

# 2. Fungsi Akuisisi Data dengan Retry
def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")

print("Mengambil data dari Overpass API...")
data = ambil_overpass(QUERY)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_path = Path(f"data/raw/osm_akomodasi_kintamani_{stamp}.json")
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Data mentah disimpan ke: {raw_path}")


In [ ]:
# 3. Meratakan (Flatten) JSON ke DataFrame
def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)
        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)

akomodasi = ratakan(data["elements"])
csv_path = Path("data/interim/akomodasi_kintamani.csv")
akomodasi.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Dataset tabular disimpan ke: {csv_path}")
